#BANCO DE VOZES PARA IDENTIFICAÇÃO

In [ ]:
# Mostra a GPU (deve aparecer "Tesla T4")
!nvidia-smi
# faster-whisper: o Whisper rápido | pyannote.audio: reconhece as vozes | yt-dlp: pega o áudio do YouTube
!pip install -q -U yt-dlp faster-whisper pyannote.audio
# Confere se o ffmpeg (conversor de áudio) está instalado
!ffmpeg -version | head -1

In [ ]:
!unzip -o /content/vozes.zip -d /content
!ls /content/*.wav

In [ ]:
import os, numpy as np, torch, subprocess
from google.colab import files, userdata
from pyannote.audio import Model, Inference

HF_TOKEN = userdata.get("HF_TOKEN")
DEVICE = torch.device("cuda")

try:
    modelo = Model.from_pretrained("pyannote/embedding", token=HF_TOKEN)
except TypeError:
    modelo = Model.from_pretrained("pyannote/embedding", use_auth_token=HF_TOKEN)

inferencia = Inference(modelo, window="whole", device=DEVICE)

# --- Funções auxiliares ---
def carregar_audio(caminho):
    """Lê qualquer áudio e converte para 16 kHz, mono (formato que os modelos esperam)."""
    saida = subprocess.run(
        ["ffmpeg", "-i", caminho, "-f", "f32le", "-ac", "1", "-ar", "16000", "-loglevel", "error", "-"],
        capture_output=True, check=True).stdout
    return np.frombuffer(saida, np.float32).copy()

def impressao_digital(audio):
    """Transforma um trecho de áudio em um vetor que representa a voz."""
    onda = torch.from_numpy(audio).unsqueeze(0)
    return inferencia({"waveform": onda, "sample_rate": 16000})

def perfil_da_voz(audio, janela_seg=3.0):
    """Divide o áudio em pedaços de 3s, tira a impressão digital de cada um e faz a média."""
    n = int(16000 * janela_seg)
    vetores = []
    for i in range(0, len(audio) - n + 1, n):
        v = impressao_digital(audio[i:i + n])
        vetores.append(v / np.linalg.norm(v))
    if not vetores:
        raise ValueError("Áudio curto demais (precisa de pelo menos 3 segundos).")
    media = np.mean(vetores, axis=0)
    return media / np.linalg.norm(media)

# ---
ARQUIVOS = {
    "Mediador": "/content/mediador.wav",
    "Bolsonaro": "/content/bolsonaro.wav" ,
    'Lula': "/content/lula.wav"
}

PERFIS = {}
for nome, caminho in ARQUIVOS.items():
    if not os.path.exists(caminho):
        print(f"\n⚠️ Não achei {caminho}. Envie manualmente o áudio de: {nome}")
        enviado = files.upload()
        caminho = list(enviado.keys())[0]
    audio = carregar_audio(caminho)
    print(f"{nome}: {len(audio) / 16000:.0f} segundos de áudio")
    PERFIS[nome] = perfil_da_voz(audio)
    print(f"✅ Voz de {nome} cadastrada!")

np.savez("/content/perfis.npz", **PERFIS)
print("\nPronto! Perfis salvos:", list(PERFIS.keys()))

#TESTE COM PRIMEIRO DEBATE (irt off)

In [ ]:
import os, glob, ctypes, site

# Carrega as bibliotecas CUDA 12 instaladas pelo pip
for base in site.getsitepackages():
    for padrao in ["nvidia/cublas/lib/libcublas.so.12*", "nvidia/cublas/lib/libcublasLt.so.12*",
                   "nvidia/cudnn/lib/libcudnn*.so.9*"]:
        for arq in glob.glob(os.path.join(base, padrao)):
            try:
                ctypes.CDLL(arq, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
import os, time
from scipy.spatial.distance import cosine
from faster_whisper import WhisperModel

# ================== CONFIGURAÇÕES (mude aqui) ==================
LIVE_URL    = "https://www.youtube.com/live/Gqc1rQtzWUk?si=0rhCmm1VC-vGxYMI"
CHUNK_SEG   = 10      # tamanho de cada pedaço de áudio (maior = mais preciso, mas mais atraso)
LIMIAR      = 0.65    # distância máxima para aceitar um nome (menor = mais rigoroso)
SIMULAR     = True   # True = testar com vídeo gravado (finge que é ao vivo)
DEBUG       = True   # True = mostra as distâncias para você ajustar o LIMIAR
ARQUIVO_TXT = "/content/transteste.txt"
# ===============================================================

# Carrega o Whisper na GPU (se ficar atrasado, troque "large-v3" por "medium")
whisper = WhisperModel("large-v3", device="cuda", compute_type="float16")

def quem_fala(audio_segmento):
    """Compara a voz do trecho com as 3 cadastradas e devolve o nome mais parecido."""
    v = impressao_digital(audio_segmento)
    v = v / np.linalg.norm(v)
    distancias = {nome: cosine(v, perfil) for nome, perfil in PERFIS.items()}
    melhor = min(distancias, key=distancias.get)
    if DEBUG:
        print("   [debug]", {k: round(d, 2) for k, d in distancias.items()})
    return melhor if distancias[melhor] <= LIMIAR else "Desconhecido"

def pegar_url_do_audio(url):
    """Pergunta ao yt-dlp qual é o endereço direto do áudio da live."""
    cmd = ["yt-dlp", "-g", "-f", "bestaudio/best", url]
    if os.path.exists("/content/cookies.txt"):          # opcional (ver seção 6)
        cmd += ["--cookies", "/content/cookies.txt"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("yt-dlp falhou:\n" + r.stderr)
    return r.stdout.strip().splitlines()[0]

def mmss(segundos):
    return f"{int(segundos // 60):02d}:{int(segundos % 60):02d}"

# --- Abre o fluxo de áudio da live ---
url_audio = pegar_url_do_audio(LIVE_URL)
cmd = ["ffmpeg", "-loglevel", "error"]
if SIMULAR:
    cmd += ["-re"]  # lê na velocidade normal, como se fosse ao vivo
cmd += ["-reconnect", "1", "-reconnect_streamed", "1", "-reconnect_delay_max", "5",
        "-i", url_audio, "-vn", "-f", "s16le", "-ac", "1", "-ar", "16000", "-"]
ffmpeg = subprocess.Popen(cmd, stdout=subprocess.PIPE)

bytes_por_pedaco = 16000 * 2 * CHUNK_SEG   # 16000 amostras/s * 2 bytes * segundos
ultimo_falante = "Desconhecido"
numero_pedaco = 0

print("🔴 Ouvindo a live... (para parar: botão ■ da célula)\n")
try:
    with open(ARQUIVO_TXT, "a", encoding="utf-8") as arq:
        while True:
            bruto = ffmpeg.stdout.read(bytes_por_pedaco)
            if len(bruto) < 16000 * 2:        # menos de 1s de áudio = a live acabou
                print("\n⏹️ A transmissão terminou.")
                break

            audio = np.frombuffer(bruto, np.int16).astype(np.float32) / 32768.0
            inicio_pedaco = numero_pedaco * CHUNK_SEG
            numero_pedaco += 1

            segmentos, _ = whisper.transcribe(
                audio, language="pt", vad_filter=True,
                beam_size=5, condition_on_previous_text=False)

            for seg in segmentos:
                texto = seg.text.strip()
                if not texto:
                    continue
                trecho = audio[int(seg.start * 16000): int(seg.end * 16000)]
                # Trechos muito curtos não dão para identificar: reaproveita o último falante
                falante = quem_fala(trecho) if len(trecho) >= 16000 else ultimo_falante
                ultimo_falante = falante

                linha = f"[{mmss(inicio_pedaco + seg.start)}] {falante}: {texto}"
                print(linha)
                arq.write(linha + "\n")
                arq.flush()

except KeyboardInterrupt:
    print("\n🛑 Interrompido por você.")
finally:
    ffmpeg.terminate()
    print(f"💾 Transcrição salva em {ARQUIVO_TXT}")

In [ ]:
!pip install -q python-docx
from docx import Document
from google.colab import files

ARQUIVO_TXT = "/content/transteste.txt"   # se estiver usando o arquivo de teste, troque por /content/transteste.txt

doc = Document()
doc.add_heading("Transcrição do debate", level=1)

with open(ARQUIVO_TXT, encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha:
            continue
        p = doc.add_paragraph()
        # Separa "[00:12] Lula: texto" em tempo, nome e fala
        if "] " in linha and ": " in linha:
            tempo, resto = linha.split("] ", 1)
            nome, fala = resto.split(": ", 1)
            p.add_run(tempo + "] ")
            p.add_run(nome + ": ").bold = True
            p.add_run(fala)
        else:
            p.add_run(linha)

doc.save("/content/transcricao.docx")
files.download("/content/transcricao.docx")

In [ ]:
!pip install -q nvidia-cublas-cu12 nvidia-cudnn-cu12

#APLICAÇÃO

In [ ]:
import os, time
from scipy.spatial.distance import cosine
from faster_whisper import WhisperModel

# ================== CONFIGURAÇÕES (mude aqui) ==================
LIVE_URL    = "https://www.youtube.com/live/Q1wRBNn2gsE?si=CUPWTzMfKWklngD5"
CHUNK_SEG   = 10      # tamanho de cada pedaço de áudio (maior = mais preciso, mas mais atraso)
LIMIAR      = 0.65    # distância máxima para aceitar um nome (menor = mais rigoroso)
SIMULAR     = False   # True = testar com vídeo gravado (finge que é ao vivo)
DEBUG       = False   # True = mostra as distâncias para você ajustar o LIMIAR
ARQUIVO_TXT = "/content/transisa.txt"
# ===============================================================

# Carrega o Whisper na GPU (se ficar atrasado, troque "large-v3" por "medium")
whisper = WhisperModel("large-v3", device="cuda", compute_type="float16")

def quem_fala(audio_segmento):
    """Compara a voz do trecho com as 3 cadastradas e devolve o nome mais parecido."""
    v = impressao_digital(audio_segmento)
    v = v / np.linalg.norm(v)
    distancias = {nome: cosine(v, perfil) for nome, perfil in PERFIS.items()}
    melhor = min(distancias, key=distancias.get)
    if DEBUG:
        print("   [debug]", {k: round(d, 2) for k, d in distancias.items()})
    return melhor if distancias[melhor] <= LIMIAR else "Desconhecido"

def pegar_url_do_audio(url):
    """Pergunta ao yt-dlp qual é o endereço direto do áudio da live."""
    cmd = ["yt-dlp", "-g", "-f", "bestaudio/best", url]
    if os.path.exists("/content/cookies.txt"):          # opcional (ver seção 6)
        cmd += ["--cookies", "/content/cookies.txt"]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("yt-dlp falhou:\n" + r.stderr)
    return r.stdout.strip().splitlines()[0]

def mmss(segundos):
    return f"{int(segundos // 60):02d}:{int(segundos % 60):02d}"

# --- Abre o fluxo de áudio da live ---
url_audio = pegar_url_do_audio(LIVE_URL)
cmd = ["ffmpeg", "-loglevel", "error"]
if SIMULAR:
    cmd += ["-re"]  # lê na velocidade normal, como se fosse ao vivo
cmd += ["-reconnect", "1", "-reconnect_streamed", "1", "-reconnect_delay_max", "5",
        "-i", url_audio, "-vn", "-f", "s16le", "-ac", "1", "-ar", "16000", "-"]
ffmpeg = subprocess.Popen(cmd, stdout=subprocess.PIPE)

bytes_por_pedaco = 16000 * 2 * CHUNK_SEG   # 16000 amostras/s * 2 bytes * segundos
ultimo_falante = "Desconhecido"
numero_pedaco = 0

print("🔴 Ouvindo a live... (para parar: botão ■ da célula)\n")
try:
    with open(ARQUIVO_TXT, "a", encoding="utf-8") as arq:
        while True:
            bruto = ffmpeg.stdout.read(bytes_por_pedaco)
            if len(bruto) < 16000 * 2:        # menos de 1s de áudio = a live acabou
                print("\n⏹️ A transmissão terminou.")
                break

            audio = np.frombuffer(bruto, np.int16).astype(np.float32) / 32768.0
            inicio_pedaco = numero_pedaco * CHUNK_SEG
            numero_pedaco += 1

            segmentos, _ = whisper.transcribe(
                audio, language="pt", vad_filter=True,
                beam_size=5, condition_on_previous_text=False)

            for seg in segmentos:
                texto = seg.text.strip()
                if not texto:
                    continue
                trecho = audio[int(seg.start * 16000): int(seg.end * 16000)]
                # Trechos muito curtos não dão para identificar: reaproveita o último falante
                falante = quem_fala(trecho) if len(trecho) >= 16000 else ultimo_falante
                ultimo_falante = falante

                linha = f"[{mmss(inicio_pedaco + seg.start)}] {falante}: {texto}"
                print(linha)
                arq.write(linha + "\n")
                arq.flush()

except KeyboardInterrupt:
    print("\n🛑 Interrompido por você.")
finally:
    ffmpeg.terminate()
    print(f"💾 Transcrição salva em {ARQUIVO_TXT}")

In [ ]:
!pip install -q python-docx
from docx import Document
from google.colab import files

ARQUIVO_TXT = "/content/transisa.txt"   # se estiver usando o arquivo de teste, troque por /content/transteste.txt

doc = Document()
doc.add_heading("Transcrição do debate", level=1)

with open(ARQUIVO_TXT, encoding="utf-8") as f:
    for linha in f:
        linha = linha.strip()
        if not linha:
            continue
        p = doc.add_paragraph()
        # Separa "[00:12] Lula: texto" em tempo, nome e fala
        if "] " in linha and ": " in linha:
            tempo, resto = linha.split("] ", 1)
            nome, fala = resto.split(": ", 1)
            p.add_run(tempo + "] ")
            p.add_run(nome + ": ").bold = True
            p.add_run(fala)
        else:
            p.add_run(linha)

doc.save("/content/transcrisa.docx")
files.download("/content/transcrisa.docx")